# 01 – Upoznavanje dokumenta i evaluacionog skupa

Cilj: videti kako izgleda dokument posle parsiranja, koje slike sadrži i koliki deo kontrolnih pitanja zavisi od slika, tabela i memoa.

In [ ]:
import sys, json
from pathlib import Path

# Koren projekta = prvi roditeljski folder koji sadrži src/ (radi bez obzira odakle se notebook pokreće)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from IPython.display import Image, display, Markdown

from config import KB_DOCX, EVAL_QUESTIONS, DATA_PROCESSED
from parse import extract_figures, docx_to_markdown

## 1. Evaluacioni skup

In [ ]:
questions = json.loads(EVAL_QUESTIONS.read_text(encoding='utf-8'))
df = pd.DataFrame(questions)
print(f'{len(df)} pitanja')
df['category'].value_counts()

In [ ]:
df['needs_figure'] = df['sources'].apply(lambda s: any(x.startswith('Figure') for x in s))
df['abstain'] = df['category'] == 'abstain'
print(f"Pitanja koja zavise od slika: {df['needs_figure'].sum()} ({df['needs_figure'].mean():.0%})")
print(f"Pitanja bez odgovora u dokumentu: {df['abstain'].sum()}")
df[['id', 'question', 'sources', 'category']]

## 2. Slike iz dokumenta

Svaka slika se vezuje za naslov *Figure N* ispod nje i za sekciju u kojoj se nalazi.

In [ ]:
figures = extract_figures(KB_DOCX, DATA_PROCESSED / 'figures')
pd.DataFrame(figures)[['figure_id', 'caption', 'section', 'source_file']]

In [ ]:
# Prikaz svih slika
for f in figures:
    display(Markdown(f"### {f['figure_id']}: {f['caption']} (sekcija {f['section']})"))
    display(Image(filename=f['image_path'], width=700))

## 3. Tekst i tabele (Docling → Markdown)

In [ ]:
md_text = docx_to_markdown(KB_DOCX, DATA_PROCESSED / 'NorthStar_Knowledge_Base.md')
lines = md_text.splitlines()
headings = [l for l in lines if l.startswith('#')]
table_rows = [l for l in lines if l.startswith('|')]
print(f'{len(md_text):,} karaktera, {len(headings)} naslova, {len(table_rows)} redova tabela, '
      f"{md_text.count('<!-- image -->')} mesta za slike")
print('\n'.join(headings[:25]))

In [ ]:
# Primer: sekcija 5.2 sa tabelom hotel cap-ova
start = md_text.index('5.2 Hotels')
print(md_text[start:start + 900])

## Zapažanja i sledeći koraci

- Sadržaj (TOC) na početku treba ukloniti pre chunking-a.
- Slike su u Markdown-u samo `<!-- image -->`: treba ih zameniti VLM opisima (dan 2).
- Memo PT-2026-04 postoji samo kao skenirana slika (Figure 7).
- Poglavlje 17 (arhiva) treba označiti kao `superseded`.
